# 02 — Vanilla Diffusion Policy Baseline

이 노트북은 실행 순서만 정의합니다. 재현성/학습/샘플링/시각화 로직은 `src/` 모듈에 있습니다.


## 1. Setup

In [ ]:
import importlib.util
import sys
from pathlib import Path

if importlib.util.find_spec('google') is not None and importlib.util.find_spec('google.colab') is not None:
    from google.colab import drive
    drive.mount('/content/drive')

REPO_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = next((p for p in REPO_ROOT_CANDIDATES if (p / 'src' / 'paths.py').exists()), None)
assert REPO_ROOT is not None, 'repo root with src/paths.py not found; run this notebook from the cloned repository'
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from paths import ARTIFACT_ROOT, DATA_DIR, CHECKPOINTS_DIR, FIGURES_DIR, ensure_artifact_dirs
ensure_artifact_dirs()


## 2. Dependencies

In [ ]:
!pip install -q -r {REPO_ROOT / 'requirements.txt'}
print('✓ requirements.txt dependencies are ready')


## 3. Imports + Config

In [ ]:
import gymnasium as gym
import torch

from configs import get_experiment_config
from experiment_plots import (
    plot_action_chunks,
    plot_frequency_sweep,
    plot_loss_curve,
    plot_sample_histogram,
    print_step3_vs_step4_table,
)
from experiment_runner import (
    apply_best_ema_after_training,
    build_model,
    build_noise_scheduler,
    load_data_and_build_loaders,
    rollout_at_frequency,
    run_frequency_sweep,
    sample_frequency_variants,
    sample_single_batch,
    train_or_load_checkpoint,
)
from phase import (
    controllability_sweep_frequencies,
    legacy_periodic_sweep_frequencies,
    periodic_offline_frequencies,
    trajectory_offline_frequencies,
    training_frequency_triplet,
)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}, device={device}')

cfg = get_experiment_config('vanilla')
train_cond_fn = cfg.resolve_train_cond_fn()
sample_cond_fn = cfg.resolve_sample_cond_fn()
print(f'Experiment config: {cfg.name} — {cfg.display_name}')


## 4. Data

In [ ]:
data, train_ds, val_ds, train_loader, val_loader = load_data_and_build_loaders(cfg, DATA_DIR)


## 5. Model + Scheduler

In [ ]:
model = build_model(cfg, data, device=device)
noise_scheduler, ns_config, NUM_INFERENCE_STEPS = build_noise_scheduler(cfg)
ema = cfg.build_ema(model)


## 6. Train or Load

In [ ]:
TRAIN = False
train_losses, val_log, best_ema_state, CKPT_PATH = train_or_load_checkpoint(
    train=TRAIN, cfg=cfg, model=model, ema=ema, noise_scheduler=noise_scheduler,
    train_loader=train_loader, val_loader=val_loader, checkpoints_dir=CHECKPOINTS_DIR, device=device,
)


## 7. Loss Curve

In [ ]:
plot_loss_curve(train_losses, val_log, FIGURES_DIR / cfg.artifacts.loss_plot_name, title=cfg.display_name)


## 8. Offline Sampling

In [ ]:
samples = sample_single_batch(
    model, ema, ns_config, val_ds, data, sample_cond_fn,
    device=device, num_inference_steps=NUM_INFERENCE_STEPS, batch_size=64, seed=data['seed'],
)
plot_sample_histogram(samples, FIGURES_DIR / 'vanilla_dp_sample_hist.png', title='Vanilla DP sampled actions')


## 9. Chunk Visualization

In [ ]:
obs = val_ds[0]['obs'].unsqueeze(0)
chunks = {}
for i in range(4):
    chunk = sample_single_batch(
        model, ema, ns_config, [val_ds[0]], data, sample_cond_fn,
        device=device, num_inference_steps=NUM_INFERENCE_STEPS, batch_size=1, seed=data['seed'] + i,
    )[0]
    chunks[f'sample {i}'] = chunk
plot_action_chunks(chunks, FIGURES_DIR / 'vanilla_dp_action_chunks.png', title='Vanilla DP — same obs, different noise seeds', act_dim=data['ACT_DIM'])


## 10. Ant Rollout

In [ ]:
env = gym.make('Ant-v5')
results = rollout_at_frequency(
    model, ema, env, ns_config, data, sample_cond_fn,
    freq_hz=float(data['freq_window_mean']), n_seeds=cfg.evaluation.n_seeds,
    max_steps=cfg.evaluation.max_steps, num_inference_steps=NUM_INFERENCE_STEPS,
    deterministic_sampling=cfg.evaluation.deterministic_sampling, device=device, dt=cfg.evaluation.dt,
)
